# SynBio 2026 — GFP ML Training Notebook (Colab)

A **self-contained, beginner-friendly** notebook that trains ML brightness predictors
on the official `GFP_data.xlsx` and produces a competition-ready submission of 6
GFP variants designed by combining literature-backed thermostabilizing mutations
with data-driven brightness mutations, then ranked by an XGBoost model.

The ML pipeline is in the same style as your QSAR notebook:

- Random Forest, Gradient Boosting, MLP, XGBoost (the QSAR model zoo)
- 10-fold cross-validation
- Grid search hyperparameters (optional)
- Best model picked by RMSE
- SHAP feature importance (optional)

What this notebook produces:

- `outputs/ml_cv_report.json` — CV metrics for every model
- `outputs/best_model.pkl` — the trained best model
- `outputs/submission.csv` — final 6 sequences in the official format
- `outputs/design_log.json` — provenance for every picked sequence

**How to use it.** Run the cells in order. Each section is self-contained — if a
cell fails, you can fix and re-run that cell without restarting.

**Section 0 only:** change `TEAM_NAME` to your real team name before generating
the final submission.


## 1. Install packages

In [ ]:
# Mount Google Drive at the very beginning (force_remount=True is safe to
# re-run; it skips errors if Drive is already mounted in this session).
#from google.colab import drive
#drive.mount('/content/drive', force_remount=True)

# Run once per Colab session.
# scikit-learn and XGBoost are the workhorses; pandas/openpyxl read the data.
!pip install -q scikit-learn xgboost pandas openpyxl tqdm
# SHAP is optional (used in section 8). Install if you want feature-importance plots.
!pip install -q shap

import sys
print('Python:', sys.version.split()[0])
import sklearn, xgboost, pandas as pd, numpy as np
print('sklearn:', sklearn.__version__, '| xgboost:', xgboost.__version__,
      '| pandas:', pd.__version__, '| numpy:', np.__version__)


## 2. Connect your competition data

Two ways to give Colab access to the official data files:

**Option A — Google Drive (recommended).** Upload `GFP_data.xlsx`,
`Exclusion_List.csv`, `AAseqs of 5 GFP proteins.txt`, and `submission_template.csv`
to a folder in your Drive (e.g. `MyDrive/SynBio2026/`), then run the cell below.

**Option B — Direct upload.** Skip the Drive cell and use the upload widget further down.


In [ ]:
# Mount Drive (force_remount avoids 'already mounted' errors on re-runs)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

# Edit DATA_DIR to wherever your competition files live in Drive.
# Common variants are tried automatically below.
DATA_DIR = '/content/drive/MyDrive/SynBio2026'

REQUIRED_BASENAMES = [
    'GFP_data.xlsx',
    'Exclusion_List.csv',
    'AAseqs of 5 GFP proteins_20260511.txt',
    'submission_template.csv',
]

def all_present(d):
    d = Path(d)
    return d.is_dir() and all((d / n).exists() for n in REQUIRED_BASENAMES)

if not all_present(DATA_DIR):
    print(f'[INFO] Files not found in {DATA_DIR}. Searching common locations ...')
    candidates = [
        '/content/drive/MyDrive/SynBio2026',
        '/content/drive/MyDrive/SynBio 2026',
        '/content/drive/MyDrive/synbio2026',
        '/content/drive/MyDrive/Synbio2026',
        '/content/drive/MyDrive/SynBio2026_data',
        '/content/drive/MyDrive',
        '/content',
    ]
    for c in candidates:
        if all_present(c):
            DATA_DIR = c
            print(f'[OK] auto-detected: {c}')
            break
    else:
        # Recursive search up to depth 3 inside MyDrive (with a small file cap for safety)
        my = Path('/content/drive/MyDrive')
        if my.exists():
            print('[INFO] scanning MyDrive (depth 3) for a folder with all four files ...')
            for d in [my] + [p for p in my.rglob('*') if p.is_dir()][:3000]:
                if all_present(d):
                    DATA_DIR = str(d)
                    print(f'[OK] found at: {d}')
                    break

DATA_DIR = Path(DATA_DIR)
print(f'\nUsing DATA_DIR = {DATA_DIR}')
if DATA_DIR.is_dir():
    print('Contents:')
    for f in sorted(DATA_DIR.iterdir()):
        print(f'  {f.name}')
else:
    print('[!] DATA_DIR is not a directory yet — see section 2b for the upload fallback.')


In [ ]:
# OPTIONAL FALLBACK: if Drive setup is hard, just upload the four files here.
# Skip this cell if section 2 already found everything.
#
# Uncomment the lines below, run, and pick all four files at once in the
# upload dialog (Cmd/Ctrl-click to multi-select).

# from google.colab import files
# uploaded = files.upload()
# DATA_DIR = Path('/content')
# print('Uploaded:', sorted(uploaded.keys()))


## 3. Verify all required files are present

In [ ]:
# Verify all four required files exist. If anything's missing, you get a
# clear error with troubleshooting hints instead of a blunt AssertionError.

required = {
    'data':       'GFP_data.xlsx',
    'exclusion':  'Exclusion_List.csv',
    'refseqs':    'AAseqs of 5 GFP proteins_20260511.txt',
    'template':   'submission_template.csv',
}

paths = {}
missing = []
for key, name in required.items():
    p = DATA_DIR / name
    if p.exists():
        paths[key] = p
        print(f'  [OK]   {name}  ({p.stat().st_size/1e6:.1f} MB)')
    else:
        missing.append(name)
        print(f'  [MISS] {name}')

if missing:
    print()
    print(f'[ERROR] {len(missing)} required file(s) missing in {DATA_DIR}.')
    print()
    print('What to do:')
    print('  1) Make sure the file names match exactly (case-sensitive, including spaces).')
    print('  2) If your files are in a different Drive folder, edit DATA_DIR in section 2.')
    print('  3) Or use the upload fallback in section 2b (uncomment the 3 lines and run).')
    print()
    if DATA_DIR.is_dir():
        print(f'Files I CAN see in {DATA_DIR}:')
        for f in sorted(DATA_DIR.iterdir()):
            print(f'  {f.name}')
    raise FileNotFoundError(f'Missing: {missing}')

print('\n[OK] all four files found.')


## 4. Configuration

Change these values to control speed vs. quality. Start in `'quick'` mode to verify
the notebook runs end-to-end (~2 min on Colab CPU), then switch to `'full'`.


In [ ]:
TEAM_NAME = 'YourTeamName'   # <-- CHANGE THIS before final submission

RUN_MODE  = 'quick'           # 'quick' (fast) | 'full' (best quality, ~10-30 min)
SEED      = 7
KFOLDS    = 10                # K for K-fold cross-validation (your QSAR style)

if RUN_MODE == 'quick':
    SUBSAMPLE_TRAIN  = 5_000
    N_CANDIDATES     = 3_000
    DO_GRIDSEARCH    = False
elif RUN_MODE == 'full':
    SUBSAMPLE_TRAIN  = None          # use all 141k rows
    N_CANDIDATES     = 10_000
    DO_GRIDSEARCH    = True
else:
    raise ValueError("RUN_MODE must be 'quick' or 'full'")

print(f'TEAM_NAME = {TEAM_NAME!r}')
print(f'RUN_MODE  = {RUN_MODE!r}')
print(f'SUBSAMPLE_TRAIN = {SUBSAMPLE_TRAIN}')
print(f'N_CANDIDATES   = {N_CANDIDATES}')
print(f'DO_GRIDSEARCH  = {DO_GRIDSEARCH}')

OUT_DIR = Path('/content/outputs')
OUT_DIR.mkdir(exist_ok=True)


## 5. Reference sequences and constants

The competition's sfGFP is **239 aa** but canonical avGFP/sfGFP literature uses
**238-aa** numbering. We handle the +1 shift around position 172 throughout.


In [ ]:
AA20 = 'ACDEFGHIKLMNPQRSTVWY'
AA_SET = set(AA20)
MIN_LEN, MAX_LEN = 220, 250

# Canonical sfGFP (238 aa, post-2026-05-27 correction) and canonical avGFP (238 aa)
SFGFP = ('MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRH'
         'DFFKSAMPEGYVQERTISFKDDGTYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIK'
         'ANFKIRHNVEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSVLSKDPNEKRDHMVLLEFVTAAGITHGMDELYK')
AVGFP = ('MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLSYGVQCFSRYPDHMKQH'
         'DFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIK'
         'VNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK')

assert len(SFGFP) == 238 and len(AVGFP) == 238

# Chromophore residues — NEVER mutate these (positions 65/66/67 in 239-numbering)
CHROMO_POS_0IDX = (64, 65, 66)

# Map canonical avGFP-numbered position → competition sfGFP (239) numbering
def canon_to_comp(pos): return pos  # identity since 2026-05-27 correction (was: pos if pos <= 171 else pos + 1)

# Physicochemical AA features for the featurizer
AA_HYDRO = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,
            'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,
            'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
AA_VOL   = {'A':88,'R':173,'N':114,'D':111,'C':108,'Q':143,'E':138,'G':60,'H':153,
            'I':166,'L':166,'K':168,'M':162,'F':189,'P':112,'S':89,'T':116,'W':227,
            'Y':193,'V':140}
AA_CHARGE = {a:0 for a in AA20}
AA_CHARGE.update({'D':-1,'E':-1,'K':1,'R':1,'H':0.1})

print('sfGFP_239 length:', len(SFGFP))
print('avGFP_238 length:', len(AVGFP))


## 6. Load brightness data and featurize

We engineer two kinds of features per mutation set:

- **Global descriptors** (10 dims): mutation count + summed Δhydrophobicity,
  Δvolume, Δcharge, and counts to charged/hydrophobic/Pro/Gly residues
- **One-hot**: one indicator per frequent (position, target-AA) seen ≥5 times

This replaces VIF feature selection (your QSAR step) — VIF doesn't scale to 141k
rows, but tree models handle correlated features natively.


In [ ]:
import re
import pandas as pd, numpy as np
from collections import Counter

WT_LABEL = 'WT'

def parse_mut_string(s):
    if not isinstance(s, str) or s.upper() == WT_LABEL or not s.strip():
        return []
    out = []
    for tok in s.split(':'):
        tok = tok.strip()
        if len(tok) < 3: continue
        try:
            out.append((tok[0], int(tok[1:-1]), tok[-1]))
        except (ValueError, IndexError):
            pass
    return out

print('Loading brightness data ...')
df = pd.read_excel(paths['data'], sheet_name='brightness')
print(f'  loaded {len(df):,} rows from {df["GFP type"].nunique()} families')
print('  per-family counts:')
print(df['GFP type'].value_counts().to_string())

# We train on avGFP (closest to sfGFP) — the largest family with ~52k rows
df_av = df[df['GFP type'] == 'avGFP'].copy()
wt_brightness = float(df_av[df_av['aaMutations'].astype(str).str.upper() == WT_LABEL]['Brightness'].mean())
df_av = df_av[df_av['aaMutations'].astype(str).str.upper() != WT_LABEL]
print(f'\navGFP samples: {len(df_av):,}  |  WT log10 brightness = {wt_brightness:.3f}')

if SUBSAMPLE_TRAIN and SUBSAMPLE_TRAIN < len(df_av):
    df_av = df_av.sample(n=SUBSAMPLE_TRAIN, random_state=SEED)
    print(f'Subsampled to {len(df_av):,} rows for fast iteration')

mutation_sets = [parse_mut_string(s) for s in df_av['aaMutations'].astype(str)]
y = df_av['Brightness'].to_numpy(dtype=np.float32)


In [ ]:
# Featurizer (matches src/ml_brightness.py:MutationFeaturizer)
class MutationFeaturizer:
    GLOBAL_DIM = 10
    def __init__(self, min_count=5, max_features=2000):
        self.min_count = min_count
        self.max_features = max_features
        self.mut_index = {}
        self.feature_names = []

    def fit(self, mutation_sets):
        c = Counter()
        for muts in mutation_sets:
            for ref, pos, alt in muts:
                c[(pos, alt)] += 1
        kept = [k for k, n in c.most_common(self.max_features) if n >= self.min_count]
        self.mut_index = {k: i for i, k in enumerate(kept)}
        self.feature_names = (
            ['n_mut','abs_dHydro','dHydro','abs_dVol','dVol','dCharge',
             'n_to_charged','n_to_hydroph','n_to_Pro','n_to_Gly']
            + [f'mut_{p}{a}' for (p, a) in kept])
        return self

    def transform(self, mutation_sets):
        n = len(mutation_sets)
        d = self.GLOBAL_DIM + len(self.mut_index)
        X = np.zeros((n, d), dtype=np.float32)
        for i, muts in enumerate(mutation_sets):
            X[i, 0] = len(muts)
            for ref, pos, alt in muts:
                if ref in AA_HYDRO and alt in AA_HYDRO:
                    dh = AA_HYDRO[alt] - AA_HYDRO[ref]
                    dv = AA_VOL[alt]  - AA_VOL[ref]
                    dq = AA_CHARGE[alt] - AA_CHARGE[ref]
                    X[i,1]+=abs(dh); X[i,2]+=dh
                    X[i,3]+=abs(dv); X[i,4]+=dv
                    X[i,5]+=dq
                    if alt in 'DEKR':    X[i,6]+=1
                    if alt in 'AVLIMFW': X[i,7]+=1
                    if alt == 'P':       X[i,8]+=1
                    if alt == 'G':       X[i,9]+=1
                idx = self.mut_index.get((pos, alt))
                if idx is not None:
                    X[i, self.GLOBAL_DIM + idx] = 1.0
        return X

    def fit_transform(self, ms): return self.fit(ms).transform(ms)


feat = MutationFeaturizer(min_count=5, max_features=2000)
X = feat.fit_transform(mutation_sets)
print(f'X shape: {X.shape}  ({feat.GLOBAL_DIM} global + {len(feat.mut_index)} one-hot)')
print(f'y shape: {y.shape}  range=[{y.min():.2f}, {y.max():.2f}]')


## 7. Train models with K-fold cross-validation

Mirrors **Part 2/3/4/5 of your QSAR notebook**: same model zoo (RF / GBR / MLP /
XGBoost), same K-fold protocol. Reports RMSE and R² across folds.

**Set `DO_GRIDSEARCH = True` in section 4** to add hyperparameter search (slow).


In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

def kfold_cv(model, X, y, k=KFOLDS, random_state=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    rmse, r2 = [], []
    for tr, te in kf.split(X):
        model.fit(X[tr], y[tr])
        yp = model.predict(X[te])
        rmse.append(float(np.sqrt(mean_squared_error(y[te], yp))))
        r2.append(float(r2_score(y[te], yp)))
    return {'rmse_mean': float(np.mean(rmse)), 'rmse_std': float(np.std(rmse)),
            'r2_mean':   float(np.mean(r2)),   'r2_std':   float(np.std(r2)),
            'rmse_per_fold': rmse, 'r2_per_fold': r2}

models = {
    'RandomForest': RandomForestRegressor(
        n_estimators=200, max_depth=20, min_samples_leaf=4,
        n_jobs=-1, random_state=SEED),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.07,
        subsample=0.85, random_state=SEED),
    'MLP': MLPRegressor(
        hidden_layer_sizes=(64, 32), activation='relu',
        solver='adam', max_iter=80, early_stopping=True, random_state=SEED),
    'XGBoost': XGBRegressor(
        n_estimators=400, max_depth=8, learning_rate=0.07,
        subsample=0.85, colsample_bytree=0.85,
        tree_method='hist', n_jobs=-1, random_state=SEED, verbosity=0),
}

results = {}
for name, mdl in models.items():
    t0 = time.time()
    cv = kfold_cv(mdl, X, y, k=KFOLDS)
    cv['time_seconds'] = time.time() - t0
    results[name] = cv
    print(f'{name:>17s}  RMSE={cv["rmse_mean"]:.3f}±{cv["rmse_std"]:.3f}  '
          f'R²={cv["r2_mean"]:+.3f}±{cv["r2_std"]:.3f}  ({cv["time_seconds"]:.1f}s)')


In [ ]:
# Display results as a nicely formatted DataFrame
metrics_df = pd.DataFrame({
    name: {'RMSE_mean': r['rmse_mean'], 'RMSE_std': r['rmse_std'],
           'R2_mean':   r['r2_mean'],   'R2_std':   r['r2_std'],
           'time_s':    r['time_seconds']}
    for name, r in results.items()
}).T.sort_values('RMSE_mean')
metrics_df


In [ ]:
# Bar chart of model performance (matplotlib only — no plotly dependency)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
m = metrics_df.sort_values('RMSE_mean')
axes[0].barh(m.index, m['RMSE_mean'], xerr=m['RMSE_std'])
axes[0].set_xlabel('RMSE (lower = better)'); axes[0].set_title('K-fold RMSE')
axes[0].invert_yaxis()
axes[1].barh(m.index, m['R2_mean'], xerr=m['R2_std'], color='C2')
axes[1].set_xlabel('R² (higher = better)'); axes[1].set_title('K-fold R²')
axes[1].invert_yaxis()
plt.tight_layout(); plt.show()


## 7b. Predicted vs Observed for each model (QSAR-style scatter plots)

For each model, we use `cross_val_predict` to get out-of-fold predictions on every
training row, then plot Predicted vs Observed. The closer the points hug the y = x line,
the better the model. R² and RMSE are annotated in each panel — same style as your
QSAR notebook's *Data visualization* cells.


In [ ]:
from sklearn.model_selection import cross_val_predict
import matplotlib.pyplot as plt

# Get out-of-fold predictions for each model
cv_preds = {}
for name, mdl in models.items():
    print(f'  predicting {name} ...')
    cv_preds[name] = cross_val_predict(mdl, X, y, cv=KFOLDS, n_jobs=-1)

# 2 columns x ceil(n/2) rows
names = list(cv_preds)
ncols = 2
nrows = (len(names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.8 * nrows))
axes = axes.ravel() if nrows > 1 else [axes] if ncols == 1 else axes

for ax, name in zip(axes, names):
    yp = cv_preds[name]
    rmse = float(np.sqrt(mean_squared_error(y, yp)))
    r2 = float(r2_score(y, yp))
    ax.scatter(y, yp, s=4, alpha=0.25)
    lo, hi = min(y.min(), yp.min()), max(y.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.2, label='y = x')
    ax.set_xlabel('Observed log10 brightness')
    ax.set_ylabel('Predicted log10 brightness')
    ax.set_title(f'{name}\nR² = {r2:.3f}   RMSE = {rmse:.3f}')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper left', fontsize=8)

# Hide any unused axes
for ax in axes[len(names):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


### Residual plots (optional but useful)

If the residuals (y_true − y_pred) trend with the predicted value, the model is
biased and may need different features or a transform. Flat-around-zero is good.


In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.0 * nrows))
axes = axes.ravel() if nrows > 1 else [axes] if ncols == 1 else axes
for ax, name in zip(axes, names):
    yp = cv_preds[name]
    res = y - yp
    ax.scatter(yp, res, s=4, alpha=0.25)
    ax.axhline(0, color='r', ls='--', lw=1)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Residual (Observed − Predicted)')
    ax.set_title(f'{name} — residuals')
    ax.grid(alpha=0.25)
for ax in axes[len(names):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()


## 7c. Feature importance for each tree model (MDI)

Mean Decrease in Impurity is the built-in importance score for tree-based models.
Mirrors the *Random feature importance (MDI)* cell in your QSAR notebook.
Higher = the feature is used more often / more decisively in the trees.


In [ ]:
tree_models = {n: m for n, m in models.items()
               if hasattr(m, 'feature_importances_')
               or n in ('RandomForest', 'GradientBoosting', 'XGBoost')}

# Make sure each tree model has been fit on the full data
for n, m in tree_models.items():
    if not hasattr(m, 'feature_importances_'):
        print(f'  fitting {n} on full training data ...')
        m.fit(X, y)

fig, axes = plt.subplots(1, len(tree_models), figsize=(5.5 * len(tree_models), 6))
if len(tree_models) == 1: axes = [axes]

TOP = 20
for ax, (name, mdl) in zip(axes, tree_models.items()):
    imp = mdl.feature_importances_
    order = np.argsort(imp)[::-1][:TOP]
    names_top = [feat.feature_names[i] for i in order][::-1]
    vals_top = imp[order][::-1]
    ax.barh(names_top, vals_top)
    ax.set_title(f'{name}\nTop-{TOP} features (MDI)')
    ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

# Tabular view of the top features for the best model
best_name_so_far = min(results, key=lambda n: results[n]['rmse_mean'])
if best_name_so_far in tree_models:
    imp = tree_models[best_name_so_far].feature_importances_
    top_df = pd.DataFrame({
        'feature': [feat.feature_names[i] for i in np.argsort(imp)[::-1][:25]],
        'importance': sorted(imp, reverse=True)[:25],
    })
    print(f'Top-25 features in {best_name_so_far}:')
    display(top_df)


## 8. (Optional) GridSearchCV for the best model

When `DO_GRIDSEARCH = True`, do a small grid search on the best model from
section 7 — same pattern as your QSAR notebook's *Search optimal hyperparameter*
cells. Skip this in `'quick'` mode.


In [ ]:
best_name = min(results, key=lambda n: results[n]['rmse_mean'])
print('Best model from CV:', best_name)

if DO_GRIDSEARCH:
    print('\nRunning grid search ...')
    if best_name == 'XGBoost':
        param_grid = {
            'n_estimators': [300, 500],
            'max_depth':    [6, 8, 10],
            'learning_rate':[0.05, 0.07, 0.10],
        }
        base = XGBRegressor(tree_method='hist', n_jobs=-1, random_state=SEED, verbosity=0)
    elif best_name == 'GradientBoosting':
        param_grid = {
            'n_estimators': [200, 400],
            'max_depth':    [4, 5, 6],
            'learning_rate':[0.05, 0.07, 0.10],
        }
        base = GradientBoostingRegressor(subsample=0.85, random_state=SEED)
    elif best_name == 'RandomForest':
        param_grid = {
            'n_estimators':     [200, 400],
            'max_depth':        [None, 20, 40],
            'min_samples_leaf': [2, 4, 8],
        }
        base = RandomForestRegressor(n_jobs=-1, random_state=SEED)
    else:  # MLP
        param_grid = {
            'hidden_layer_sizes': [(64,32), (128,64), (32,16)],
            'alpha':              [1e-4, 1e-3],
        }
        base = MLPRegressor(activation='relu', solver='adam',
                            max_iter=80, early_stopping=True, random_state=SEED)
    gs = GridSearchCV(base, param_grid, cv=5, scoring='neg_root_mean_squared_error',
                      n_jobs=-1, verbose=1)
    gs.fit(X, y)
    print('Best params:', gs.best_params_)
    print('Best CV RMSE:', -gs.best_score_)
    best_model = gs.best_estimator_
else:
    best_model = models[best_name]
    print('(Skipping grid search; using default hyperparameters)')


In [ ]:
# Refit best model on the full dataset and save
import pickle
best_model.fit(X, y)
bundle = {
    'model':          best_model,
    'featurizer':     feat,
    'model_name':     best_name,
    'wt_brightness':  wt_brightness,
    'family':         'avGFP',
    'cv_results':     results,
}
model_path = OUT_DIR / 'best_model.pkl'
with model_path.open('wb') as f:
    pickle.dump(bundle, f)
print(f'Saved → {model_path}  ({model_path.stat().st_size/1e6:.2f} MB)')

import json
(OUT_DIR / 'ml_cv_report.json').write_text(json.dumps(
    {'family': 'avGFP', 'n_samples': int(len(y)), 'wt_brightness': wt_brightness,
     'feature_dim': int(X.shape[1]), 'k': KFOLDS, 'best_model': best_name,
     'models': results}, indent=2))
print(f'Saved → {OUT_DIR / "ml_cv_report.json"}')


## 9. (Optional) SHAP feature importance

SHAP tells you which features (which mutations) the model relies on most for its
predictions. Mirrors the *SHAP analysis* cell of your QSAR notebook.


In [ ]:
try:
    import shap
    bg = X[np.random.RandomState(SEED).choice(len(X), size=100, replace=False)]
    expl = shap.TreeExplainer(best_model, data=bg)
    sample = X[np.random.RandomState(SEED+1).choice(len(X), size=200, replace=False)]
    sv = expl.shap_values(sample)
    mean_abs = np.abs(sv).mean(axis=0)
    order = np.argsort(mean_abs)[::-1][:25]
    top_df = pd.DataFrame({
        'feature':    [feat.feature_names[i] for i in order],
        'mean_|SHAP|': mean_abs[order]
    })
    print('Top-25 most important features:')
    display(top_df)
    shap.summary_plot(sv, sample, feature_names=feat.feature_names, max_display=20)
except Exception as e:
    print('SHAP analysis skipped:', e)


## 10. Build the design pipeline

Now we use the trained model to design new sequences.

**Mutation pools** (same logic as our `src/`):

- **Thermostability candidates** — 16 literature-validated mutations from sfGFP,
  superfolder, TGP, mClover3, etc. (no Cys; cell-free is reducing).
- **Brightness candidates** — top-2 substitutions per position from the data,
  filtered for functional retention ≥ 50% and best-case Δlog10 ≥ 0.05.


In [ ]:
# Thermostability candidates (positions in canonical 238-aa numbering)
def make_thermo_candidates():
    raw = [
        # (canonical_pos, target_aa, weight, source)
        (206,'K',0.95,'A206K monomerizing'),
        (167,'T',0.85,'I167T folding'),
        (149,'K',0.65,'surface charge'),
        (147,'T',0.55,'surface stabilizing'),
        (177,'F',0.55,'TGP-inspired core packing'),
        (223,'I',0.60,'TGP barrel packing'),
        (224,'L',0.55,'TGP barrel packing'),
        (46, 'L',0.50,'F46L loop'),
        (80, 'R',0.50,'Q80R surface'),
        (231,'L',0.45,'T231L barrel'),
        (78, 'K',0.40,'Q78K surface'),
        (176,'K',0.40,'N176K surface'),
        (154,'L',0.45,'TGP-like core'),
        (43, 'L',0.35,'F43L loop'),
        (141,'L',0.30,'I141L barrel'),
        (195,'I',0.30,'V195I core'),
    ]
    return [(canon_to_comp(p), a, w) for p, a, w in raw
            if (canon_to_comp(p) - 1) not in CHROMO_POS_0IDX]

# Brightness candidates from data (top per-position substitutions)
def make_brightness_candidates(df_av_full, mutation_sets, y, top_k=2,
                               min_func_rate=0.5, min_support=5, min_effect=0.05):
    """Top mutations per position with positive top-quartile effect AND high functional retention."""
    from collections import defaultdict
    bucket = defaultdict(list)
    for muts, b in zip(mutation_sets, y):
        for ref, pos, alt in muts:
            bucket[(pos, alt)].append(float(b))
    table = {}
    for key, vals in bucket.items():
        if len(vals) < min_support: continue
        vs = sorted(vals, reverse=True)
        top_q = vs[:max(1, len(vs)//4)]
        top_mean = sum(top_q)/len(top_q)
        func = sum(1 for v in vals if v >= 2.5)/len(vals)
        if func < min_func_rate: continue
        table[key] = {'effect': top_mean - wt_brightness, 'func_rate': func}
    # Pick top_k per position with effect >= min_effect, no Cys
    per_pos = defaultdict(list)
    for (pos, aa), v in table.items():
        if aa == 'C': continue
        if v['effect'] < min_effect: continue
        per_pos[pos].append((aa, v['effect']))
    pool = []
    for pos, items in per_pos.items():
        items.sort(key=lambda t: -t[1])
        for aa, eff in items[:top_k]:
            comp_pos = canon_to_comp(pos)
            if (comp_pos - 1) in CHROMO_POS_0IDX: continue
            pool.append((comp_pos, aa, eff))
    return pool

thermo_pool   = make_thermo_candidates()
bright_pool   = make_brightness_candidates(df_av, mutation_sets, y)
print(f'Thermo pool:    {len(thermo_pool)} candidates')
print(f'Brightness pool: {len(bright_pool)} candidates')

# Merge, dedup by (pos, aa), keep highest weight
seen = {}
for pos, aa, w in thermo_pool + bright_pool:
    key = (pos, aa)
    if key not in seen or w > seen[key][2]:
        seen[key] = (pos, aa, w)
merged_pool = list(seen.values())
print(f'Merged pool:    {len(merged_pool)} unique (pos, aa) candidates')


## 11. Generate the candidate library

In [ ]:
import random

def apply_mutations(parent, muts):
    s = list(parent)
    for pos, aa in muts:
        i = pos - 1
        if 0 <= i < len(s): s[i] = aa
    return ''.join(s)

def random_combos(parent, pool, n, size_range=(2, 8), seed=SEED):
    rng = random.Random(seed)
    out = set()
    attempts = 0
    while len(out) < n and attempts < n * 50:
        attempts += 1
        k = rng.randint(*size_range)
        k = min(k, len(pool))
        combo = rng.sample(pool, k)
        by_pos = {}
        for pos, aa in combo:
            by_pos[pos] = aa
        out.add(apply_mutations(parent, list(by_pos.items())))
    return list(out)

# Use sfGFP_239 as primary parent (competition WT)
pool_sfgfp = [(p, a) for p, a, _ in merged_pool]
candidates = random_combos(SFGFP, pool_sfgfp, n=N_CANDIDATES, seed=SEED)
candidates = [SFGFP] + candidates  # always include WT for reference
print(f'Generated {len(candidates):,} candidate sequences')


## 12. Hard filters: validity + exclusion list

Three rules from the competition spec:

- length 220–250
- starts with M, only standard 20 AAs, no stop codons
- not in `Exclusion_List.csv` (135k+ entries)


In [ ]:
def valid(seq):
    if not isinstance(seq, str): return False
    if not (MIN_LEN <= len(seq) <= MAX_LEN): return False
    if seq[0] != 'M': return False
    if set(seq) - AA_SET: return False
    if '*' in seq: return False
    return True

before = len(candidates)
candidates = [s for s in candidates if valid(s)]
print(f'After validity filter: {len(candidates):,} (removed {before-len(candidates):,})')

# Exclusion-list filter (hash-based for O(1) lookup)
import csv, hashlib
print('Loading exclusion list ...')
excl = set()
with paths['exclusion'].open() as f:
    r = csv.reader(f); next(r, None)
    for row in r:
        if row and row[0].strip():
            excl.add(hashlib.sha256(row[0].strip().upper().encode()).hexdigest())
print(f'  {len(excl):,} excluded sequences loaded')

before = len(candidates)
candidates = [s for s in candidates
              if hashlib.sha256(s.strip().upper().encode()).hexdigest() not in excl]
print(f'After exclusion filter: {len(candidates):,} (removed {before-len(candidates):,})')


## 13. Score every candidate with the ML model + thermostability prior

Composite score = `0.6 * normalized(ML brightness)` + `0.4 * normalized(thermo prior)`


In [ ]:
# Convert sequences → mutation sets relative to sfGFP_239 → features → ML predictions
def seq_to_mut_set(seq, parent=SFGFP):
    if len(seq) != len(parent): return []
    out = []
    is_239_legacy = (len(parent) == 239)  # legacy guard
    for pos, (a, b) in enumerate(zip(parent, seq), 1):
        if a == b: continue
        if is_239_legacy:
            if pos <= 171:    canon = pos
            elif pos == 172:  continue
            else:             canon = pos - 1
        else:
            canon = pos
        out.append((a, canon, b))
    return out

cand_mut_sets = [seq_to_mut_set(s) for s in candidates]
X_cand = feat.transform(cand_mut_sets)
ml_scores = best_model.predict(X_cand)
print(f'ML predictions:  range=[{ml_scores.min():.2f}, {ml_scores.max():.2f}]')

# Thermostability prior (sum of literature weights for matched mutations)
thermo_lookup = {(p, a): w for p, a, w in thermo_pool}
def thermo_score(seq):
    if len(seq) != len(SFGFP): return 0.0
    s = 0.0
    for i, (a, b) in enumerate(zip(SFGFP, seq), 1):
        if a == b: continue
        w = thermo_lookup.get((i, b))
        if w is not None: s += w
        else: s -= 0.05  # small drift penalty
    return s

thermo_scores = np.array([thermo_score(s) for s in candidates], dtype=np.float32)
print(f'Thermo scores:   range=[{thermo_scores.min():.2f}, {thermo_scores.max():.2f}]')

# Composite score
def normalize(v):
    lo, hi = v.min(), v.max()
    return (v - lo) / (hi - lo + 1e-9)

W_ML, W_THERMO = 0.6, 0.4
composite = W_ML * normalize(ml_scores) + W_THERMO * normalize(thermo_scores)
print(f'Composite range: [{composite.min():.3f}, {composite.max():.3f}]')


## 14. Pick the diverse Top 6

In [ ]:
def hamming(a, b):
    return sum(1 for x, y in zip(a, b) if x != y) + abs(len(a) - len(b))

# Sort by composite score, then greedy pick with Hamming-distance diversity
order = np.argsort(-composite)
picks_idx = []
MIN_HAMMING = 4
for i in order:
    if len(picks_idx) >= 6: break
    cand_seq = candidates[i]
    if all(hamming(cand_seq, candidates[j]) >= MIN_HAMMING for j in picks_idx):
        picks_idx.append(int(i))

# Build picks DataFrame for inspection
rows = []
for rank, i in enumerate(picks_idx, 1):
    seq = candidates[i]
    muts = [f'{a}{p}{b}' for (a,p,b) in seq_to_mut_set(seq)]
    rows.append({
        'rank':       rank,
        'n_mut':      len(muts),
        'ml_pred':    float(ml_scores[i]),
        'thermo':     float(thermo_scores[i]),
        'composite':  float(composite[i]),
        'mutations':  ','.join(muts),
        'sequence':   seq,
    })
picks_df = pd.DataFrame(rows)
display(picks_df.drop(columns=['sequence']))


## 15. Final validation & write submission

In [ ]:
# Independent re-validation
final_seqs = picks_df['sequence'].tolist()
assert len(final_seqs) == 6, 'Expected 6 sequences'
for i, s in enumerate(final_seqs, 1):
    assert valid(s), f'Sequence {i} failed validation'
    h = hashlib.sha256(s.strip().upper().encode()).hexdigest()
    assert h not in excl, f'Sequence {i} is in the exclusion list!'
print('[OK] all 6 sequences pass validation and are not in the exclusion list')

# Verify pairwise diversity
print('\nPairwise Hamming distances:')
for i in range(6):
    for j in range(i+1, 6):
        print(f'  [{i+1}] vs [{j+1}]: {hamming(final_seqs[i], final_seqs[j])}')

# Write submission CSV in the official format
sub_path = OUT_DIR / 'submission.csv'
with sub_path.open('w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Team_Name', 'Seq_ID', 'Sequence'])
    for i, s in enumerate(final_seqs, 1):
        w.writerow([TEAM_NAME, i, s])
print(f'\nWrote submission → {sub_path}')

# Verify header matches the official template byte-for-byte
import csv as _csv
with paths['template'].open() as f:
    tpl_header = next(_csv.reader(f))
with sub_path.open() as f:
    sub_header = next(_csv.reader(f))
assert sub_header == tpl_header, f'Header mismatch: {sub_header} vs {tpl_header}'
print('[OK] header matches submission_template.csv exactly')


In [ ]:
# Write design log with full provenance
design_log = {
    'meta': {
        'team_name':      TEAM_NAME,
        'run_mode':       RUN_MODE,
        'seed':           SEED,
        'n_candidates':   N_CANDIDATES,
        'best_ml_model':  best_name,
        'ml_cv_rmse':     results[best_name]['rmse_mean'],
        'ml_cv_r2':       results[best_name]['r2_mean'],
        'weights':        {'ml': W_ML, 'thermo': W_THERMO},
    },
    'designs': [],
}
for r in rows:
    design_log['designs'].append({
        'seq_id':           r['rank'],
        'n_mutations':      r['n_mut'],
        'mutations':        r['mutations'].split(',') if r['mutations'] else [],
        'ml_brightness_log10':  round(r['ml_pred'], 4),
        'thermostability_prior': round(r['thermo'], 4),
        'composite_score':  round(r['composite'], 4),
        'sequence':         r['sequence'],
    })
log_path = OUT_DIR / 'design_log.json'
log_path.write_text(json.dumps(design_log, indent=2))
print(f'Wrote design log → {log_path}')


## 16. Download outputs

In [ ]:
# Show all output files
print('Output files:')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1e3:.1f} KB)')

# Trigger downloads (Colab only)
try:
    from google.colab import files
    for fname in ['submission.csv', 'design_log.json', 'ml_cv_report.json', 'best_model.pkl']:
        p = OUT_DIR / fname
        if p.exists():
            files.download(str(p))
except Exception as e:
    print('(Not in Colab, or download cancelled:', e, ')')


## 17. What to do next

**Tuning to improve ranking**

- Increase `SUBSAMPLE_TRAIN` (set to `None` for full 141k) — improves ML R² substantially.
- Set `DO_GRIDSEARCH = True` (section 4) and re-run.
- Adjust `W_ML` vs `W_THERMO` in section 13. Higher `W_THERMO` → more conservative
  literature-mutation picks; higher `W_ML` → more data-driven picks.
- Add ESM-2 features: encode sequences with `facebook/esm2_t6_8M_UR50D`, then
  concatenate the embedding to the engineered features. Re-train.

**Before submitting**

1. Set `TEAM_NAME` to your real team name.
2. Run with `RUN_MODE = 'full'` and `DO_GRIDSEARCH = True`.
3. Re-run sections 14–15 to regenerate the submission.
4. Open `submission.csv` in a text editor to confirm UTF-8, Unix line endings.
5. Push your code to GitHub (public repo with this notebook) — required by the
   competition.

**Reference: which other models to add later**

- **ESM-2 / ESM-3 / SaProt**: use as feature extractor, then run the same QSAR
  regressor zoo on the embeddings.
- **ProteinMPNN**: generates structure-preserving sequence variants. Add its
  outputs to the candidate library before scoring.
- **Tranception**: use its zero-shot fitness as an additional input feature.
